# PathSnap — Phase 5 to Phase 7: Five-Model Benchmarking & Optuna Bayesian Optimization
**Course:** UE24CS352A – Machine Learning | PES University  
**Role:** Member 2 — Machine Learning Modeling & Hyperparameter Optimization Lead  

### Objectives:
1. **Phase 5 (Five Baseline Models):** Implement pipelines for 5 distinct algorithm families:
   - **Logistic Regression:** Linear / probabilistic baseline + StandardScaler + balanced weights
   - **k-Nearest Neighbors (k-NN):** Instance-based metric similarity (weights='distance', n_neighbors=15)
   - **Decision Tree:** Tree-based nonlinear rules (max_depth=12, min_samples_leaf=10)
   - **Random Forest:** Ensemble bagging (n_estimators=300, max_depth=18)
   - **HistGradientBoosting:** Sequential gradient boosting (learning_rate=0.08, max_iter=200)
2. **Group-Aware Cross-Validation:** 5-fold `GroupKFold` grouped strictly by `trip_id` (zero data leakage across trips).
3. **Phase 6 (Model Evaluation & Selection):**
   - Candidate-level binary Precision, Recall, F1, and Log Loss
   - Observation-level Top-1 Accuracy (primary classifier map-matching measure)
   - Training runtime & inference latency per 1,000 observations
   - Frozen evaluation on all 10 `student_test` trips
4. **Phase 7 (Bayesian Optimization):**
   - Optuna TPE search over learning rate, tree depth, leaf count, and regularizations
   - Tuned vs. Untuned comparison table & trial history
5. **Artifact Export:** Save `candidate_ranker.joblib` and `metadata.json` for API deployment.

In [ ]:
# Cell 00: Imports & Global Configuration
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import optuna
import joblib

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR, MODELS_DIR
from src.models.train import get_model_pipelines, run_group_kfold_cv, evaluate_student_test, FEATURE_COLS
from src.models.bayesian_opt import run_bayesian_optimization

SEED = 42
np.random.seed(SEED)
print("PathSnap Member 2 Modeling Environment Ready.")

In [ ]:
# Cell 01: Load Processed Candidate Feature Tables
train_path = PROCESSED_DATA_DIR / "train_candidates.parquet"
test_path = PROCESSED_DATA_DIR / "student_test_candidates.parquet"

df_train = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

print(f"Train candidates: {df_train.shape} across {df_train['trip_id'].nunique()} trips")
print(f"Test candidates:  {df_test.shape} across {df_test['trip_id'].nunique()} trips")
print("\nFeature Columns:", FEATURE_COLS)
df_train.head()

In [ ]:
# Cell 02: Run 5-Fold GroupKFold Cross-Validation
df_folds, df_summary = run_group_kfold_cv(df_train, feature_cols=FEATURE_COLS, n_splits=5)
print("\n--- 5-Model GroupKFold Cross-Validation Summary ---")
display(df_summary)

In [ ]:
# Cell 03: Visualize CV Benchmark Metrics
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=df_summary, x="Model", y="Top1_Accuracy_Mean", palette="viridis")
plt.title("Observation Top-1 Accuracy (5-Fold GroupKFold CV)")
plt.ylabel("Top-1 Accuracy")
plt.xticks(rotation=30)

plt.subplot(1, 2, 2)
sns.barplot(data=df_summary, x="Model", y="Candidate_F1_Mean", palette="magma")
plt.title("Candidate Binary F1-Score (5-Fold GroupKFold CV)")
plt.ylabel("Binary F1")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 04: Frozen Evaluation on 10 Student-Test Trips
df_test_res = evaluate_student_test(df_train, df_test, feature_cols=FEATURE_COLS)
print("\n--- Student-Test Performance Summary ---")
display(df_test_res)

In [ ]:
# Cell 05: Optuna Bayesian Hyperparameter Optimization
opt_results = run_bayesian_optimization(
    df_train=df_train,
    df_test=df_test,
    feature_cols=FEATURE_COLS,
    n_trials=20
)
print("Best Hyperparameters:", opt_results["best_params"])
display(opt_results["comparison"])

In [ ]:
# Cell 06: Inspect Exported Model Artifacts
with open(MODELS_DIR / "metadata.json", "r") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))
print(f"\nModel artifact exists at: {MODELS_DIR / 'candidate_ranker.joblib'}")